# 06 訊號、閾值與 7-1-7 — 筆記本

8 月的 Santa Luz：鉤端螺旋體病的疫情正在減退，積水卻沒有退，*Aedes*（斑蚊）正忙著繁殖。Maya 每個週一的問題很簡單：**這週的登革熱病例數是異常，還是只是雨季？** 在這本筆記本裡你會：

1. 讀入每週登革熱序列（`data/synthetic/ph/santa_luz_dengue_weekly.csv`，**合成資料**）；
2. 建立**流行閾值**——平常年份同一週的平均值 + 2 個標準差——並找出 2026 年第一個越過它的週別；
3. 畫出**地方性流行通道**；
4. 看看為什麼最新幾週總是偏低（遲到的通報）；
5. 在 40 起過去的事件（`signals_717.csv`）上計算 **7-1-7 時效**，並解讀是什麼拖慢了它們。

這裡每一個病例數都是虛構的；方法卻是真實的。

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## 步驟 1 — 每週序列

每一列是一個**流行病學週（morbidity week）**。這個檔案使用 ISO 週（週一到週日，標示成 `2026-W32`）；`iso_year` 與 `iso_week` 是分開的欄位，這樣我們才能把不同年份的「同一週」對齊。我們來讀入它，並看看 2026 年到目前為止的情況。

In [ ]:
import statistics

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -- CJK 字型設定（避免中文標籤顯示成方框 □□□）--
import pathlib
import matplotlib.font_manager as fm

for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

# 動態偵測實際註冊的 CJK 字型名稱（避免 .ttc face 0 陷阱）
_discovered = []
for _entry in fm.fontManager.ttflist:
    _nlower = _entry.name.lower()
    if any(_kw in _nlower for _kw in ("cjk", "wenquanyi", "wqy")):
        if _entry.name not in _discovered:
            _discovered.append(_entry.name)
_preferred = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
_cjk_fonts = list(_discovered)
for _n in _preferred:
    if _n not in _cjk_fonts:
        _cjk_fonts.append(_n)
plt.rcParams["font.sans-serif"] = _cjk_fonts + [
    f for f in plt.rcParams.get("font.sans-serif", []) if f not in _cjk_fonts
]
plt.rcParams["axes.unicode_minus"] = False
del _discovered, _preferred, _cjk_fonts

weekly = pd.read_csv("data/synthetic/ph/santa_luz_dengue_weekly.csv", parse_dates=["week_start"])
print(weekly.shape, "| 年份：", sorted(weekly["iso_year"].unique()))
print("每年的週數：", weekly.groupby("iso_year").size().to_dict())

y2026 = weekly[weekly["iso_year"] == 2026].set_index("iso_week")
print("\n2026 年，第 28–39 週：")
print(y2026.loc[28:39, ["morbidity_week", "week_start", "cases"]].to_string())

## 步驟 2 — 建立閾值

CESU 對這個序列使用的規則（寫在資料的 README 裡）：

* 針對 ISO 週 *w*，取**基準年 2022–2025** 的第 *w*−1、*w*、*w*+1 週的病例數——4 年 × 3 週 = **12 個數字**；
* 閾值是它們的**平均值 + 2 個標準差**（樣本標準差）；
* 排除 2019 年，因為它是流行年；排除 2020–2021 年，因為 COVID-19 擾亂了就醫行為與通報。基準必須描述*平常*的年份。

> **問問 Tala：** *我有一個 pandas DataFrame `weekly`，欄位有 `iso_year`、`iso_week`（1–53）與 `cases`。虛構的範例列：`(2022, 31, 9)`、`(2022, 32, 12)`、`(2023, 32, 7)`。請對 1 到 53 的每個 ISO 週 w，計算流行閾值 = 只取 2022–2025 年第 w−1、w、w+1 週 `cases` 的平均值 + 2 × 樣本標準差（第 1 週的窗口要接到第 52 週）。請回傳以週別為索引的 Series。然後告訴我 2026 年第一個病例數高於閾值的週別。程式碼請寫得讓初學者看得懂。*

In [ ]:
BASELINE_YEARS = [2022, 2023, 2024, 2025]   # 只用平常的年份：不含 2019、2020、2021，絕對不含 2026


def window_weeks(week):
    # 第 w-1、w、w+1 週；第 1 週接到 52，第 52 週接到 1（第 53 週用 52、53、1）
    if week == 53:
        return {52, 53, 1}
    return {(week - 2) % 52 + 1, week, week % 52 + 1}


def epidemic_threshold(weekly, years):
    base = weekly[weekly["iso_year"].isin(years)]
    rows = {}
    for week in range(1, 54):
        values = base.loc[base["iso_week"].isin(window_weeks(week)), "cases"]
        rows[week] = {"n_values": len(values), "baseline_mean": values.mean(),
                      "threshold": values.mean() + 2 * values.std(ddof=1)}
    return pd.DataFrame(rows).T


thr = epidemic_threshold(weekly, BASELINE_YEARS)
season = y2026[["cases"]].join(thr)
season["above"] = season["cases"] > season["threshold"]
print(season.loc[28:39].round(1).to_string())

first_week = season.index[season["above"]].min()
print(f"\n2026 年第一個高於閾值的週別：W{first_week} "
      f"（{y2026.loc[first_week, 'week_start']:%m/%d} 起）：{season.loc[first_week, 'cases']:.0f} 例，"
      f"閾值為 {season.loc[first_week, 'threshold']:.1f}")

**檢查一下。** Doc Mon 會做的三項檢查：

1. **手算其中一週。** 列出 W32 的 12 個基準值，並用 Python 的 `statistics` 模組計算平均值 + 2 個標準差——這是獨立於 pandas 之外的第二種方法。
2. **窗口的分母。** 每一週都必須剛好有 12 個值。
3. **故事對得上。** 第一次越過閾值之前，沒有任何 2026 年的週別高於閾值，而之後每一週都高於。

In [ ]:
base = weekly[weekly["iso_year"].isin(BASELINE_YEARS)]
w32_values = base.loc[base["iso_week"].isin([31, 32, 33]), "cases"].tolist()
print("W32 的 12 個基準值：", w32_values)
by_hand = statistics.mean(w32_values) + 2 * statistics.stdev(w32_values)
print(f"手算的平均值 + 2 個標準差：{by_hand:.2f}  | 函式算出的：{thr.loc[32, 'threshold']:.2f}")

assert len(w32_values) == 12
assert abs(by_hand - thr.loc[32, "threshold"]) < 1e-9
assert (thr.loc[1:52, "n_values"] == 12).all()
assert not season.loc[:first_week - 1, "above"].any(), "更早的週別都不可以高於閾值"
assert season.loc[first_week:39, "above"].all()
print("所有檢查都通過：警報會在 W32 響起，並持續到 W39。")

## 步驟 3 — 地方性流行通道

**地方性流行通道**（endemic channel）畫出一年中每一週的「平常水準」，讓我們能拿今年的曲線去對照。我們把每個基準年畫成一條細灰線，再畫基準平均值（預期值）與閾值，最後把 2026 年疊在上面。

In [ ]:
weeks = np.arange(1, 53)
fig, ax = plt.subplots(figsize=(11, 4.2))
for year in BASELINE_YEARS:
    one = weekly[weekly["iso_year"] == year].set_index("iso_week")["cases"]
    ax.plot(one.index, one.values, color="#aeb7c0", linewidth=0.8)
ax.plot(weeks, thr.loc[weeks, "baseline_mean"], color="#2f7d32", linewidth=2, label="基準平均值（2022-2025）")
ax.fill_between(weeks, 0, thr.loc[weeks, "threshold"], color="#2f7d32", alpha=0.10)
ax.plot(weeks, thr.loc[weeks, "threshold"], color="#b45309", linewidth=2, linestyle="--",
        label="流行閾值（平均值 + 2 個標準差）")
bar_colours = np.where(season["above"], "#CE1126", "#0038A8")
ax.bar(season.index, season["cases"], color=bar_colours, alpha=0.85, label="2026 年（紅色 = 高於閾值）")
ax.annotate("颱風 Amihan\n（W29）", xy=(29, 6), xytext=(18, 45), arrowprops={"arrowstyle": "->"})
ax.set_xlabel("ISO 週")
ax.set_ylabel("每週登革熱病例數")
ax.set_title("Santa Luz 市（合成資料）：2026 年登革熱與其地方性流行通道")
ax.set_xlim(0.5, 52.5)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Tala 搞錯了：把自己墊高閾值的流行

Maya 的第一個提示說「使用檔案裡所有的年份」。Tala 照做了——於是閾值包含了 **2026 年本身**（還有 2019 年，一個流行年）。把流行放進它自己的基準，會同時墊高平均值與標準差，於是閾值跟著曲線一起升高。下面的儲存格展示了這個效果，以及一行本來就能抓出它的 `assert`。

In [ ]:
all_years = sorted(weekly["iso_year"].unique())          # Tala 的版本：2019 ... 2026
thr_wrong = epidemic_threshold(weekly, all_years)
compare = y2026.loc[30:39, ["cases"]].copy()
compare["right_threshold"] = thr.loc[30:39, "threshold"]
compare["wrong_threshold"] = thr_wrong.loc[30:39, "threshold"]
compare["alarm_right"] = compare["cases"] > compare["right_threshold"]
compare["alarm_wrong"] = compare["cases"] > compare["wrong_threshold"]
print(compare.round(1).to_string())
print(f"\nW32–W39 被標為警報的週數：正確的基準 {int(compare.loc[32:, 'alarm_right'].sum())} / 8，"
      f"Tala 的基準 {int(compare.loc[32:, 'alarm_wrong'].sum())} / 8")
silenced = compare.index[compare["alarm_right"] & ~compare["alarm_wrong"]].tolist()
print(f"W32 的閾值從 {thr.loc[32, 'threshold']:.1f} 升到 {thr_wrong.loc[32, 'threshold']:.1f}；"
      f"流行還在爬升的 {silenced} 週，警報卻沒有響")

# 能抓出問題的把關：你正在判斷的那一季，絕不能出現在它自己的基準裡
def check_baseline(years, season_year=2026, epidemic_years=(2019,)):
    assert season_year not in years, f"{season_year} 出現在它自己的基準裡"
    assert not set(epidemic_years) & set(years), "基準裡有流行年"

check_baseline(BASELINE_YEARS)            # 靜靜通過
try:
    check_baseline(all_years)
except AssertionError as err:
    print("抓到了：", err)

## 步驟 4 — 為什麼最新幾週總是偏低

每週的檔案是依**發病週**計算病例，但病例要在幾天後才會到 CESU（先就醫，再通報）。所以在任何一個週三，剛結束的那一週都還不完整，進行中的那一週看起來幾乎是空的。我們可以用[第 04 章](../04_time_place_person.md)的造冊重現這一點：只用某個日期**之前已通報**的列，依發病週計算登革熱病例數。

In [ ]:
ll = pd.read_csv("data/synthetic/ph/santa_luz_linelist.csv", parse_dates=["date_onset", "date_report"])
dengue = ll[ll["disease"] == "dengue"].copy()
dengue["onset_week"] = dengue["date_onset"].dt.isocalendar().week.astype(int)
delay_days = (dengue["date_report"] - dengue["date_onset"]).dt.days
print(f"登革熱，發病 -> 通報：中位數 {delay_days.median():.0f} 天，90% 在 {delay_days.quantile(0.9):.0f} 天內")

snapshots = {}
for as_of in ["2026-09-02", "2026-09-16", "2026-09-30"]:
    known = dengue[dengue["date_report"] <= as_of]
    snapshots[f"{int(as_of[5:7])}/{int(as_of[8:])} 時已知"] = known.groupby("onset_week").size()
snap = pd.DataFrame(snapshots).reindex(range(30, 40)).fillna(0).astype(int)
snap.index = [f"W{w}" for w in snap.index]
print(snap.T.to_string())

ax = snap.plot(kind="bar", rot=0, figsize=(10, 3.8), color=["#aeb7c0", "#FF9F1C", "#0038A8"], width=0.8)
ax.set_xlabel("發病週（2026 年）")
ax.set_ylabel("登革熱病例數")
ax.set_title("同樣幾週，在三個不同的日子來看")
plt.tight_layout()
plt.show()

## 步驟 5 — 7-1-7：應變有多快？

**7-1-7** 是每一起公共衛生事件的時效目標：

* **7** — 在疫情出現（第一個病例發病或事件開始）後 **7 天內**偵測到；
* **1** — 在偵測後 **1 天內**通報公共衛生主管機關（這裡是 PESU）；
* **7** — 在通報後 **7 天內**完成早期應變。

`signals_717.csv` 收錄 40 起合成事件，時間從 2025 年 1 月到 2026 年 8 月，每起事件有四個日期，以及一個 `bottleneck` 欄位，說明每個沒達成的目標是什麼原因造成的。

> **問問 Tala：** *我有一個 DataFrame `events`，日期欄位有 `date_emergence`、`date_detection`、`date_notification`、`date_response_complete`（虛構的一列：2025-03-01、2025-03-05、2025-03-06、2025-03-12）。請加上三個以整天計的延遲欄位（偵測 − 發生、通報 − 偵測、應變 − 通報）、三個對應 7-1-7 目標（≤ 7、≤ 1、≤ 7）的 True/False 欄位，以及一個「三項都達成」的欄位。接著印出每個目標各有多少事件達成，包含件數與百分比。*

In [ ]:
events = pd.read_csv(
    "data/synthetic/ph/signals_717.csv",
    parse_dates=["date_emergence", "date_detection", "date_notification", "date_response_complete"],
)
events["days_to_detect"] = (events["date_detection"] - events["date_emergence"]).dt.days
events["days_to_notify"] = (events["date_notification"] - events["date_detection"]).dt.days
events["days_to_respond"] = (events["date_response_complete"] - events["date_notification"]).dt.days

events["detect_ok"] = events["days_to_detect"] <= 7
events["notify_ok"] = events["days_to_notify"] <= 1
events["respond_ok"] = events["days_to_respond"] <= 7
events["all_ok"] = events[["detect_ok", "notify_ok", "respond_ok"]].all(axis=1)

targets = ["detect_ok", "notify_ok", "respond_ok", "all_ok"]
summary = pd.DataFrame({
    "met": events[targets].sum(),
    "missed": (~events[targets]).sum(),
    "percent_met": (events[targets].mean() * 100).round(0),
})
print(summary.to_string())
print("\n延遲的中位數（天）：", events[["days_to_detect", "days_to_notify", "days_to_respond"]].median().to_dict())

**檢查一下。** 手算一起事件。EV-040 是故事裡 Barangay Baybay 的登革熱暴增：疫情出現在閾值那一週的週一。讀出它的四個日期，在紙上相減，再和 Tala 的欄位比對。然後檢查「三項皆達成」的件數絕不可能多過任何單一目標。

In [ ]:
ev = events.set_index("event_id").loc["EV-040"]
print(ev[["disease", "barangay", "date_emergence", "date_detection", "date_notification", "date_response_complete"]])
print("延遲：", ev["days_to_detect"], ev["days_to_notify"], ev["days_to_respond"])

assert ev["days_to_detect"] == (pd.Timestamp("2026-08-10") - pd.Timestamp("2026-08-03")).days == 7
assert ev["detect_ok"] and ev["notify_ok"] and not ev["respond_ok"]
assert summary.loc["all_ok", "met"] <= summary.loc[["detect_ok", "notify_ok", "respond_ok"], "met"].min()
assert (events[["days_to_detect", "days_to_notify", "days_to_respond"]] >= 0).all().all(), "出現負的延遲 = 某個日期打錯了"
print("檢查通過。")

## 步驟 6 — 讀懂 bottleneck 欄位

百分比告訴你某個目標*多常*沒達成；`bottleneck` 欄位則告訴你*為什麼*。一起事件可能有好幾個原因，以 `"; "` 串在一起，所以我們把它們切開、計數，並看每一起沒達成的事件失敗在哪一個目標。這就是 Maya 帶去地方衛生委員會的清單：用經費、一位後備 DSO 或一份公文就能解決的問題。

In [ ]:
missed = events[events["bottleneck"] != "none"]
causes = missed["bottleneck"].str.split("; ").explode().str.strip()
print(f"{len(missed)} 起事件至少有一項目標沒達成；共記錄 {len(causes)} 個原因。\n")
print(causes.value_counts().head(6).to_string())

print("\n每項目標未達成的件數：", (~events[["detect_ok", "notify_ok", "respond_ok"]]).sum().to_dict())
print("\n2026 年的兩起 Santa Luz 事件：")
print(events.tail(2)[["event_id", "disease", "barangay", "days_to_detect", "days_to_notify",
                      "days_to_respond", "bottleneck"]].to_string(index=False))

## 換你試試

1. **更嚴格的閾值。** 把閾值重新算成平均值 + **1.5** 個標準差（複製 `epidemic_threshold`，改一個數字）。現在 2026 年第一個高於它的是哪一週？這會讓 2022–2025 年多出多少次誤報？*（提示：用和 `season` 一樣的方式，把每個基準年與 `thr` 合併，再計算高於閾值的週數。）*
2. **只用同一週。** 只使用基準年的第 *w* 週（不用 *w*−1、*w*、*w*+1）。這樣每週有幾個值？第一次越過閾值的週別會移動嗎？*（提示：只要改 `window_weeks`。）*
3. **哪一種疾病最慢？** 依 `disease` 對 `events` 分組，計算 `days_to_detect` 的中位數。你會先訓練 BHW 認出哪一種疾病？

In [ ]:
# 1. 更嚴格的閾值（起手式）
def epidemic_threshold_k(weekly, years, k):
    base = weekly[weekly["iso_year"].isin(years)]
    out = {}
    for week in range(1, 54):
        values = base.loc[base["iso_week"].isin(window_weeks(week)), "cases"]
        out[week] = values.mean() + k * values.std(ddof=1)
    return pd.Series(out)

thr_15 = epidemic_threshold_k(weekly, BASELINE_YEARS, 1.5)
print("2026 年第一個高於平均值 + 1.5 個標準差的週別：", (y2026["cases"] > thr_15.loc[y2026.index]).idxmax())

# 3. 偵測最慢的疾病（起手式）
print(events.groupby("disease")["days_to_detect"].median().sort_values(ascending=False).head(3))

## 你完成了什麼

* 只用**平常的**基準年建立流行閾值，手算驗證，並找出第一次越過的週別（2026-W32）。
* 畫出地方性流行通道，並抓出一個把流行本身放進去的基準。
* 看到遲到的通報如何讓最新幾週看起來偏低——把最新的長條讀成「至少這麼多」。
* 計算 7-1-7，並把 `bottleneck` 欄位變成一份待辦清單。

下一步：[第 07 章](../07_forecast_timesfm.md)會問，接下來八週可能會發生什麼事。